# Results section 2, and panels a and b of Figure 3

## Setup

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

# package root = the folder that holds Data/
def _root(start=None):
    here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd().resolve()
    for c in (here, *here.parents):
        if (c / "Data" / "conversation_turns.csv").exists():
            return c
    raise RuntimeError(f"cannot find the package root (no Data/conversation_turns.csv at or above {here})")

ROOT   = _root()
CLEAN  = ROOT / "Data"
RESDIR = ROOT / "Code" / "_output" / "results"; RESDIR.mkdir(parents=True, exist_ok=True)

### Statistical helpers

In [2]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

def glmm(formula, data, crossed, label=""):
    # crossed random intercepts, variational Bayes
    vc = {g: f"0 + C({g})" for g in crossed}
    # seed the variational fit
    np.random.seed(2026)
    m = BinomialBayesMixedGLM.from_formula(formula, vc, data).fit_vb()
    if label:
        print(f"  {label}")
    print(f"    {'term':<18}{'b':>9}{'SD':>8}{'95% CrI':>21}{'odds':>8}")
    for i, nm in enumerate(m.model.exog_names):
        lo, hi = m.fe_mean[i] - 1.96 * m.fe_sd[i], m.fe_mean[i] + 1.96 * m.fe_sd[i]
        print(f"    {nm:<18}{m.fe_mean[i]:>9.3f}{m.fe_sd[i]:>8.3f}"
              f"   [{lo:+.3f}, {hi:+.3f}]{np.exp(m.fe_mean[i]):>8.2f}")
    print("    random intercepts (SD, logit scale): "
          + "  ".join(f"{n} {np.exp(v):.2f}" for n, v in zip(m.model.vcp_names, m.vcp_mean)))
    return m

def paired_effect(x, y, unit_label, names=("first", "second")):
    d = np.asarray(y) - np.asarray(x)
    n = len(d); m, sd = d.mean(), d.std(ddof=1)
    se = sd / np.sqrt(n)
    tc = stats.t.ppf(0.975, n - 1)
    t = stats.ttest_rel(y, x)
    print(f"    {names[1]} - {names[0]} = {m:+.3f}   95% CI "
          f"[{m - tc*se:+.3f}, {m + tc*se:+.3f}]   "
          f"t({n-1}) = {t.statistic:+.2f}, p = {t.pvalue:.1e}")
    print(f"    Cohen's dz = {m / sd:+.2f}   ({unit_label}, n = {n})")
    return m, m / sd

## 1 · Materials and individual recall

In [3]:
# stimuli and recalls
STIMULI = pd.read_csv(CLEAN / "stimuli.csv")
_st = STIMULI[~STIMULI.is_lure]
ID2CAT   = {int(i): int(l) for i, l in zip(_st.item_id, _st.list)}

RECALL = pd.read_csv(CLEAN / "individual_recall.csv")
pre = RECALL[(RECALL.phase == "pre") & (RECALL.kind == "studied")].sort_values(
    ["participant", "output_position"])
PRIVATE = {str(pid): [int(i) for i in g.item_id] for pid, g in pre.groupby("participant")}
# each person's pre-conversation store: word ids per category
OWN_IDS = {pid: {} for pid in PRIVATE}
for pid, v in PRIVATE.items():
    for i in v:
        OWN_IDS[pid].setdefault(ID2CAT[i], set()).add(i)
print(f"{len(ID2CAT)} studied words in {_st.list.nunique()} categories")
print(f"  {len(PRIVATE)} first individual recalls")

100 studied words in 10 categories
  170 first individual recalls


In [4]:
IPATCH = pd.read_csv(RESDIR / "individual_patches.csv")
IPATCH["pid"] = IPATCH.pid.astype(str)

## 2 · Conversation turns

In [5]:
TURNS = pd.read_csv(CLEAN / "conversation_turns.csv")
CONVS = pd.read_csv(CLEAN / "conversations.csv")

rows = []
for cid, g in TURNS.sort_values(["conversation", "turn_index"]).groupby("conversation"):
    roles = list(dict.fromkeys(g.speaker_role))
    pid = {r: str(g.participant[g.speaker_role == r].iloc[0]) for r in roles}
    nth = {r: int(g.nth_conversation_for_speaker[g.speaker_role == r].iloc[0]) for r in roles}
    rows.append(dict(
        network=str(g.network.iloc[0]), dyad=g.dyad.iloc[0], topology=g.topology.iloc[0],
        round=int(g["round"].iloc[0]), nth=nth, pid=pid,
        speakers=list(g.speaker_role),
        kinds=list(g.kind),
        cats=[None if pd.isna(v) else int(v) for v in g.list],
        words=[None if pd.isna(v) else int(v) for v in g.item_id],
        raw=list(g.text)))
STREAM = pd.DataFrame(rows)
STREAM["n_turns"] = STREAM.kinds.apply(len)
STREAM.to_pickle(RESDIR / "stream.pkl")

print(f"{len(STREAM)} conversations, {STREAM.n_turns.mean():.1f} turns each")
print(f"  {STREAM.pid.apply(lambda d: tuple(d.values())).explode().nunique()} people, "
      f"{STREAM.network.nunique()} networks")
ex = CONVS[~CONVS.included]
print(f"\n  excluded logs: {len(ex)} of {len(CONVS)}")
for _, r in ex.iterrows():
    print(f"    network {r.network:>2}  dyad {r.dyad}   "
          f"{r.n_list_turns} of {r.n_turns} turns were word lists")

250 conversations, 42.0 turns each
  170 people, 17 networks

  excluded logs: 5 of 255
    network 14  dyad DE   4 of 11 turns were word lists
    network 14  dyad EF   8 of 11 turns were word lists
    network 16  dyad DE   6 of 20 turns were word lists
    network 16  dyad EF   8 of 17 turns were word lists
    network 16  dyad FH   6 of 22 turns were word lists


## 3 · Conversational patches

In [6]:
def patch_blocks(cats, kinds):
    # a run of consecutive turns sharing a category; None never joins a run
    blocks, cur = [], []
    for i, c in enumerate(cats):
        if c is None:
            if cur:
                blocks.append(cur); cur = []
            continue
        if cur and cats[cur[-1]] == c:
            cur.append(i)
        else:
            if cur:
                blocks.append(cur)
            cur = [i]
    if cur:
        blocks.append(cur)
    return blocks

rows = []
for _, r in STREAM.iterrows():
    cats, kinds, spk, pid = r.cats, r.kinds, r.speakers, r.pid
    if any(v is None for v in pid.values()):
        continue
    B = patch_blocks(cats, kinds)
    for n, block in enumerate(B):
        start = block[0]
        cat, opener = cats[start], spk[start]
        other = [x for x in pid if x != opener][0]
        # stores at the first handover, excluding the opening word
        eaten = {r.words[j] for j in range(start + 1)
                 if cats[j] == cat and r.words[j] is not None}
        held = {who: OWN_IDS.get(pid[who], {}).get(cat, set()) for who in pid}
        after = block[-1] + 1
        # censored if there is no next turn or the opener speaks again
        nxt = start + 1
        decided = nxt < len(cats) and spk[nxt] != opener
        rows.append(dict(
            network=r.network, conversation=f"{r.network}_{r.dyad}", category=str(cat),
            opener=pid[opener], partner=pid[other], position=n, start=start,
            nth_opener=r.nth.get(opener), nth_partner=r.nth.get(other),
            length=len(block),
            n_studied=sum(1 for j in block if kinds[j] == "studied"),
            n_lure=sum(1 for j in block if kinds[j] == "lure"),
            survives=int(len(block) > 1) if decided else np.nan,
            own_held=len(held[opener]), partner_held=len(held[other]),
            own_left=len(held[opener] - eaten),
            partner_left=len(held[other] - eaten),
            ended_by=(kinds[after] if after < len(cats) else "end of conversation"),
            ended_by_partner=int(after < len(cats) and spk[after] != opener),
            next_cat=(str(cats[after]) if after < len(cats) and cats[after] is not None
                      else None),
            next_by=("partner" if after < len(cats) and spk[after] != opener
                     else "opener" if after < len(cats) else None)))
PATCH = pd.DataFrame(rows)

## 4 · Continuation alone and in conversation (Figure 3a)

In [7]:
# alone: the opener decides continuation; in conversation: the responder decides
alone_p = IPATCH.groupby("pid").survives.mean()
follow = PATCH.groupby("partner").survives.mean()
paired = pd.concat([alone_p.rename("alone"), follow.rename("talking")], axis=1).dropna()
print(f"{len(paired)} people with decisions in both settings\n")

NET_OF_P = {}
for _, r in STREAM.iterrows():
    for v in r.pid.values():
        NET_OF_P[str(v)] = str(r.network)
BOTH = pd.concat([
    IPATCH.assign(talking=0)[["pid", "category", "survives", "talking"]],
    PATCH.assign(talking=1).rename(columns={"partner": "pid"})[
        ["pid", "category", "survives", "talking"]]], ignore_index=True).dropna(
    subset=["survives"])
BOTH["pid"] = BOTH.pid.astype(str); BOTH["category"] = BOTH.category.astype(str)
BOTH["network"] = BOTH.pid.map(NET_OF_P)
assert BOTH.network.notna().all()
_ = glmm("survives ~ talking", BOTH, ["pid", "category", "network"],
         "continuation, alone vs in conversation")

170 people with decisions in both settings

  continuation, alone vs in conversation
    term                      b      SD              95% CrI    odds
    Intercept             0.064   0.026   [+0.014, +0.115]    1.07
    talking              -1.510   0.033   [-1.574, -1.446]    0.22
    random intercepts (SD, logit scale): pid 0.70  category 0.41  network 0.07


## 5 · Shifted-sequence null (Figure 3b; Supplementary Note 3)

In [8]:
def mean_len(cats, kinds):
    B = patch_blocks(cats, kinds)
    return float(np.mean([len(b) for b in B])) if B else np.nan

def adj_rate(cats):
    ok = [(a, b) for a, b in zip(cats, cats[1:])]
    return float(np.mean([a is not None and b is not None and a == b for a, b in ok]))

rows = []
for _, r in STREAM.iterrows():
    cats, kinds, spk = list(r.cats), list(r.kinds), np.array(r.speakers)
    if len(set(spk)) < 2 or len(cats) < 10:
        continue
    obs_len, obs_adj = mean_len(cats, kinds), adj_rate(cats)
    # shift: speaker sequence and each person's own order kept, one rolled against the other
    who = list(dict.fromkeys(spk)); a = who[0]
    slots = np.where(spk == a)[0]
    own_c = [cats[i] for i in slots]; own_k = [kinds[i] for i in slots]
    sh_l, sh_a = [], []
    for s in range(1, len(slots)):
        c2, k2 = list(cats), list(kinds)
        for t, i in enumerate(slots):
            c2[i] = own_c[(t + s) % len(slots)]
            k2[i] = own_k[(t + s) % len(slots)]
        sh_l.append(mean_len(c2, k2)); sh_a.append(adj_rate(c2))
    if not sh_l:
        continue
    rows.append(dict(network=r.network, conversation=f"{r.network}_{r.dyad}",
                     n=len(cats), length=obs_len, adj=obs_adj,
                     length_shift=float(np.mean(sh_l)), adj_shift=float(np.mean(sh_a))))
NULLS = pd.DataFrame(rows)
print(f"{len(NULLS)} conversations; shift null averaged over every possible shift\n")
print("  mean patch depth")
print(f"    observed                     {NULLS.length.mean():.3f}")
print(f"    each person's own order,")
print(f"      rolled against the other   {NULLS.length_shift.mean():.3f}\n")
bn = NULLS.groupby("network")[["length", "length_shift"]].mean()
print("  against the shift null, network as the unit")
_ = paired_effect(bn.length_shift, bn.length, "network as the unit", ("null", "observed"))

250 conversations; shift null averaged over every possible shift

  mean patch depth
    observed                     1.567
    each person's own order,
      rolled against the other   1.120

  against the shift null, network as the unit
    observed - null = +0.446   95% CI [+0.341, +0.551]   t(16) = +9.01, p = 1.2e-07
    Cohen's dz = +2.18   (network as the unit, n = 17)


In [9]:
ba = NULLS.groupby("network")[["adj", "adj_shift"]].mean()
print("  same-category adjacency rate, network as the unit")
print(f"    observed {ba.adj.mean():.3f}   shifted apart {ba.adj_shift.mean():.3f}")
print("\n  against the shift null")
_ = paired_effect(ba.adj_shift, ba.adj, "network as the unit", ("shifted apart", "observed"))
print(f"\n  conversations above their own shift null: "
      f"{(NULLS.adj > NULLS.adj_shift).mean():.1%}")
pp = (STREAM.assign(conversation=STREAM.network + "_" + STREAM.dyad)
      .merge(NULLS[["conversation", "adj", "adj_shift"]], on="conversation"))
# the excess separately in each participant's first, second and third conversation
_pn = []
for _, r in pp.iterrows():
    for role, who in r.pid.items():
        if who is not None and r.nth.get(role) is not None:
            _pn.append(dict(pid=str(who), nth=int(r.nth[role]), excess=float(r.adj - r.adj_shift)))
_pn = pd.DataFrame(_pn)
print("\n  excess over the shift null by the participant's conversation, person as the unit")
for k_, g_ in _pn.groupby("nth"):
    print(f"    conversation {k_}: {100 * g_.excess.mean():.1f} percentage points "
          f"(SD {100 * g_.excess.std(ddof=1):.1f}), n = {len(g_)}")

  same-category adjacency rate, network as the unit
    observed 0.282   shifted apart 0.082

  against the shift null
    observed - shifted apart = +0.200   95% CI [+0.163, +0.237]   t(16) = +11.50, p = 3.8e-09
    Cohen's dz = +2.79   (network as the unit, n = 17)

  conversations above their own shift null: 96.0%

  excess over the shift null by the participant's conversation, person as the unit
    conversation 1: 19.2 percentage points (SD 16.4), n = 166
    conversation 2: 19.4 percentage points (SD 12.4), n = 164
    conversation 3: 21.4 percentage points (SD 16.2), n = 170


## 6 · Data for Figure 3a-b

In [10]:
# data for Figure 3a-b, drawn in results3_first_handover_cost.ipynb
MERGED = RESDIR / "merged"; MERGED.mkdir(parents=True, exist_ok=True)

# panel a: one value per participant, alone vs in conversation
paired.to_csv(MERGED / "f3_patch_continues.csv", index=False)
print(f"  panel a: {len(paired)} participants with patches in both settings")
print(f"    individual recall {paired.alone.mean():.3f} (SD {paired.alone.std(ddof=1):.3f})   "
      f"conversation {paired.talking.mean():.3f} (SD {paired.talking.std(ddof=1):.3f})")

# panel b: one value per conversation, observed vs shift null
NULLS[["adj", "adj_shift"]].to_csv(MERGED / "f3_coincidence.csv", index=False)
print(f"  panel b: {len(NULLS)} conversations")
_tpc = TURNS.groupby("conversation").size()
print(f"  turns per conversation: {_tpc.mean():.1f} (SD {_tpc.std():.1f}) over {len(_tpc)} conversations")

  panel a: 170 participants with patches in both settings
    individual recall 0.540 (SD 0.192)   conversation 0.233 (SD 0.167)
  panel b: 250 conversations
  turns per conversation: 42.0 (SD 9.7) over 250 conversations
